# Calculate mortality at each grid point using central estimates only

$M(x, y) = POP(x, y) \; \times \; BMR_c \; \times \; AF(x, y)  $

This script calculates gridpoint mortality for each mortality health outcome. The final step calculates the sum of the six outcomes to estimate total PM2.5 mortality

In [1]:
import os
import glob
import numpy as np
import xarray as xr
from utils.mortality_utils import mortality
import config
from utils.utils import require_dir
import pathlib

In [3]:
# === Path config ===
BMR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR")
#POP_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SSP_pop" / "SSP2")
MASKS_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country", "country masks")

In [ ]:
# Load country masks
mask_file = "GBD_Country_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataarray(mask_path)

# Load population file
pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)
pop = population.reindex_like(masks, method="nearest", tolerance=1e-9)

In [4]:
# TMREL from GBD 2021
TMREL = 4.15  # central estimate [95% Uniform CI 2.4 – 5.9]

In [5]:
# === Health variables ===
# COPD, DIABETES, ISCHEMIC_HEART_DISEASE, LOWER_RESPIRATORY_INFECTIONS,
# LUNG_CANCER, STROKE, DEMENTIA
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER",
               "STROKE", "DEMENTIA"]

In [ ]:
# === Scenario and path config ===
# For RR curves and file name
GBD_version = "GBD23"

scenarios = ["H", "HL", "L", "LN", "M", "ML", "VL"]
years = [2040, 2060, 2080, 2100]

RR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / GBD_version / "RR_curves")
PM25_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SHERPA" / "processed")
SAVE_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "mortality" / "gridpoint_mortality")

# === Main loop ===
for health_VAR in health_vars:
    print(f"Processing mortality outcome {health_VAR}")
    pattern = os.path.join(RR_DIR, f"IHME_GBD_20{GBD_version[-2:]}_AIR_POLLUTION_*_PM_RR_{health_VAR}.nc")
    matches = glob.glob(pattern)
    if len(matches) == 0:
        raise FileNotFoundError(f"No .nc file found for variable: {health_VAR}")
    if len(matches) > 1:
        raise ValueError(f"Multiple .nc files matched for variable {health_VAR}: {matches}")
    RR_values = xr.open_dataset(matches[0])["mean"]

    # --- Scale the RR to the TMREL so that RR below the TMREL=1 ---
    # Updated GBD23 risk curves are log(RR), non updated curves are RR
    if RR_values[0] == 1:
        print("Data starts at 1 so they are 'Relative Risk'")
        # Calculate log(RR)
        logRR = np.log(RR_values)
        if np.any(logRR < 0) is True:
            raise ValueError("Values of logRR < 0, should start at 0")
        # Find the log(RR) at the TMREL
        logRR_tmrel = logRR.sel(exposure=TMREL, method="nearest")
    elif RR_values[0] == 0:
        print("Data starts at 0 so they are 'log(Relative Risk)'")
        # Data is already in logRR format
        logRR = RR_values
        if np.any(logRR < 0) is True:
            raise ValueError("Values of logRR < 0, should start at 0")
        # Find the log(RR) at the TMREL
        logRR_tmrel = logRR.sel(exposure=TMREL, method="nearest")
    else:
        raise ValueError(f"Data has unknown start value: {RR_values[0]}")

    # Shift the function by the log(RR) at the TMREL (logRR_tmrel) so that log(RR)=0 at TMREL
    logRR_shifted = logRR - logRR_tmrel

    # Set log(RR) below TMREL as 0 and exponentiate to get RR
    scaled_RR = np.exp(logRR_shifted.where(logRR_shifted["exposure"] >= TMREL, 0))

    bmr_file = f"{GBD_version}_BMR_European_Country_Map_{health_VAR}_2015-2019.nc"
    bmr_path = os.path.join(BMR_DIR, bmr_file)
    BMR = xr.open_dataarray(bmr_path)  # central estimate

    for scenario in scenarios:
        for year in years:
            print(f"Processing {scenario}, year {year}, {health_VAR}")

            pm25_file = f"EU_concentration_{scenario}_{year}.nc"
            pm25_path = os.path.join(PM25_DIR, pm25_file)
            pm25 = xr.open_dataarray(pm25_path)

            POP = pop.sel(year=year)

            # Find RR at each grid point
            RR = scaled_RR.interp(exposure=pm25)
            # Calculate the attributable fraction
            AF = 1 - (1/RR)

            # Calculate mortality at each grid point
            M = mortality(AF, BMR, POP)

            description = (f"Total {health_VAR} mortality due to PM2.5 using "
                           "central estimates only - scripts "
                           "by A.F. Wells (2026)")
            M.attrs["description"] = description
            M.attrs["GBD version"] = GBD_version
            M.attrs["health_var"] = health_VAR
            M.attrs["scenario"] = scenario
            M.attrs["year"] = year

            out_file = f"Mortality_{GBD_version}_{health_VAR}_{scenario}_{year}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)

            print(f"Saving mortality to {out_path}")
            M.to_netcdf(out_path)

print("All processing complete.")

## Save the sum of all mortality outcomes

In [ ]:
for scenario in scenarios:
    for year in years:
        # Find all files for this year
        in_files = f"Mortality_{GBD_version}_*_{scenario}_{year}.nc"
        in_path = os.path.join(SAVE_DIR, in_files)
        files = sorted(glob.glob(in_path))

        # Open and combine
        datasets = [xr.open_dataarray(f) for f in files]

        # Align (important in case of slight coordinate mismatches)
        aligned = xr.align(*datasets, join="exact")

        # Sum across the health variables
        summed_da = sum(aligned)

        description = ("Total mortality due to PM2.5 using "
                       "central estimates only - scripts "
                       "by A.F. Wells (2026)")
        summed_da.attrs["description"] = description
        summed_da.attrs["GBD version"] = GBD_version
        summed_da.attrs["year"] = year
        summed_da.attrs["scenario"] = scenario

        out_file = f"Mortality_{GBD_version}_{scenario}_{year}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving summed mortality timeseries to {out_path}")
        summed_da.to_netcdf(out_path)

print("All processing complete.")